In [7]:
import pandas as pd
import numpy as np

In [34]:
patient_id = "009"

In [42]:
eeg_tags = [
    "FP1", "FP2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", 
    "F7", "F8", "P7", "P8", "FZ", "CZ", "PZ", "FC1", "FC2", "CP1", 
    "CP2", "FC5", "FC6", "CP5", "CP6"
]

# patient dependent channel order
bio_tags = {
    "001": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
    "002": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
    "003": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
    "004": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
    "005": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
    "006": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
    "007": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
    "008": {
        "OFF_1": ["EMG-RTA", "EMG-LTA", "IO", "ECG", "EMG-RGS"],
        "OFF_2": ["EMG-RTA", "EMG-RGS", "IO", "ECG", "EMG-LTA"],
    },
    "009": ["EMG-LTA", "EMG-RTA", "IO", "EMG-RGS", "ECG"],
    "010": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
    "011": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"],
    "012": ["EMG-LTA", "EMG-RTA", "IO", "ECG", "EMG-RGS"]
}

fixed_bio_tags = ["EMG-RTA", "EMG-LTA", "EMG-RGS", "IO", "ECG"]

acc_feats = [
    "ACCX", "ACCY", "ACCZ", "GYRO-X", "GYRO-Y", "GYRO-Z", "NC/SC"
]

acc_sensors = [
    "LShank-", "RShank-", "Waist-", "Arm-"
]

acc_tags = [sensor+feat for sensor in acc_sensors for feat in acc_feats]

columns = (
    ["timestamp"] +
    [f"EEG-{tag}" for tag in eeg_tags] +
    [f"{tag}" for tag in bio_tags[patient_id]] +  
    [f"{tag}" for tag in acc_tags] +
    ["label"]
)

fixed_columns = (
    ["timestamp"] +
    [f"EEG-{tag}" for tag in eeg_tags] +
    fixed_bio_tags +  
    [f"{tag}" for tag in acc_tags] +
    ["label"]
)


In [ ]:
# =========================
# LEITURA
# =========================
df = pd.read_csv("data/bronze/009/task_1.txt", header=None, names=columns, index_col=None)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99001 entries, 0 to 99000
Data columns (total 60 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      99001 non-null  object 
 1   EEG-FP1        99001 non-null  float64
 2   EEG-FP2        99001 non-null  float64
 3   EEG-F3         99001 non-null  float64
 4   EEG-F4         99001 non-null  float64
 5   EEG-C3         99001 non-null  float64
 6   EEG-C4         99001 non-null  float64
 7   EEG-P3         99001 non-null  float64
 8   EEG-P4         99001 non-null  float64
 9   EEG-O1         99001 non-null  float64
 10  EEG-O2         99001 non-null  float64
 11  EEG-F7         99001 non-null  float64
 12  EEG-F8         99001 non-null  float64
 13  EEG-P7         99001 non-null  float64
 14  EEG-P8         99001 non-null  float64
 15  EEG-FZ         99001 non-null  float64
 16  EEG-CZ         99001 non-null  float64
 17  EEG-PZ         99001 non-null  float64
 18  EEG-FC1    

In [37]:
for col in df.columns:
    if col == "timestamp":
        df[col] = pd.to_timedelta(df["timestamp"])
    elif col != "sample_id":
        df[col] = df[col].astype(np.float32) 

df.to_parquet(
    "data/bronze_parquet/001/task_1.parquet", 
    engine="pyarrow",
    compression="snappy",
)

In [38]:
df2 = pd.read_parquet("data/bronze_parquet/001/task_1.parquet")
df3 = pd.read_parquet("data/bronze_parquet/001/task_3.parquet")
df2.columns == df3.columns

array([ True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True])

In [46]:
df4 = df2[fixed_columns]
(df4["ECG"] == df2["ECG"]).describe()

count     99001
unique        1
top        True
freq      99001
Name: ECG, dtype: object

In [39]:
df2.columns

Index(['timestamp', 'EEG-FP1', 'EEG-FP2', 'EEG-F3', 'EEG-F4', 'EEG-C3',
       'EEG-C4', 'EEG-P3', 'EEG-P4', 'EEG-O1', 'EEG-O2', 'EEG-F7', 'EEG-F8',
       'EEG-P7', 'EEG-P8', 'EEG-FZ', 'EEG-CZ', 'EEG-PZ', 'EEG-FC1', 'EEG-FC2',
       'EEG-CP1', 'EEG-CP2', 'EEG-FC5', 'EEG-FC6', 'EEG-CP5', 'EEG-CP6',
       'EMG-LTA', 'EMG-RTA', 'IO', 'EMG-RGS', 'ECG', 'LShank-ACCX',
       'LShank-ACCY', 'LShank-ACCZ', 'LShank-GYRO-X', 'LShank-GYRO-Y',
       'LShank-GYRO-Z', 'LShank-NC/SC', 'RShank-ACCX', 'RShank-ACCY',
       'RShank-ACCZ', 'RShank-GYRO-X', 'RShank-GYRO-Y', 'RShank-GYRO-Z',
       'RShank-NC/SC', 'Waist-ACCX', 'Waist-ACCY', 'Waist-ACCZ',
       'Waist-GYRO-X', 'Waist-GYRO-Y', 'Waist-GYRO-Z', 'Waist-NC/SC',
       'Arm-ACCX', 'Arm-ACCY', 'Arm-ACCZ', 'Arm-GYRO-X', 'Arm-GYRO-Y',
       'Arm-GYRO-Z', 'Arm-NC/SC', 'label'],
      dtype='object')

In [47]:
df3

,timestamp,EEG_FP1,EEG_FP2,EEG_F3,EEG_F4,EEG_C3,EEG_C4,EEG_P3,EEG_P4,EEG_O1,...,Waist-GYRO-Z,Waist-NC/SC,Arm-ACCX,Arm-ACCY,Arm-ACCZ,Arm-GYRO-X,Arm-GYRO-Y,Arm-GYRO-Z,Arm-NC/SC,label
0,0 days 12:20:09,-8.8819,-9.4853,-7.7914,-4.7279,-5.2308,-1.4067,-3.0101,-6.6327,-1.4085,...,4.000000,53.0,4908.000000,-5220.000000,4058.000000,-379.000000,-6.000000,-81.000000,1802.000000,0.0
1,0 days 12:20:09.002000,-8.7917,-8.7733,-7.3297,-5.2449,-4.5153,-2.1972,-2.4093,-6.6065,-0.9058,...,5.277832,53.0,4954.931152,-5170.891602,4140.316895,-349.970886,10.863688,-76.705177,1801.977783,0.0
2,0 days 12:20:09.004000,-9.3481,-8.8580,-7.4682,-6.3635,-4.1764,-3.6429,-2.2597,-7.0374,-0.7109,...,7.151829,53.0,4994.231934,-5110.178223,4185.639160,-334.764679,30.269173,-76.023842,1801.930420,0.0
3,0 days 12:20:09.006000,-10.1607,-9.5634,-7.9150,-7.8191,-4.1835,-5.3859,-2.3987,-7.6926,-0.5972,...,9.286911,53.0,5020.767090,-5048.919434,4203.202637,-330.072998,51.342812,-77.889915,1801.894165,0.0
4,0 days 12:20:09.008000,-11.0262,-10.6876,-8.4154,-9.4878,-4.4466,-7.2297,-2.6464,-8.5900,-0.5498,...,11.347995,53.0,5029.401855,-4998.173340,4202.244141,-332.587555,73.210968,-81.237328,1801.905273,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14496,0 days 12:20:37.992000,-6.1797,-10.0270,-6.1603,-5.3816,-4.3678,-5.5278,-1.4138,-10.3033,-9.1409,...,19.764626,53.0,4640.999023,-4861.680176,3704.782227,23.877110,64.927612,-19.012220,1717.000854,0.0
14497,0 days 12:20:37.994000,-8.2978,-12.1965,-8.0980,-7.2193,-5.8668,-7.6341,-2.2675,-12.0575,-9.8887,...,19.384785,53.0,4650.764160,-4819.183594,3734.388184,30.282646,61.882153,-16.857418,1717.001587,0.0
14498,0 days 12:20:37.996000,-10.3023,-14.1670,-9.7870,-8.8630,-6.9424,-9.1117,-2.2335,-13.3422,-8.6526,...,18.922632,53.0,4659.229492,-4778.146484,3760.602783,34.849628,59.172882,-14.696506,1717.002075,0.0
14499,0 days 12:20:37.998000,-11.0582,-14.6092,-10.2371,-9.2900,-6.7846,-9.1427,-0.8467,-13.3182,-5.1593,...,18.440319,53.0,4665.329590,-4743.206543,3782.211670,37.211071,57.109074,-12.690395,1717.001709,0.0


In [48]:
[f"EEG-{tag}" for tag in eeg_tags]

['EEG-FP1',
 'EEG-FP2',
 'EEG-F3',
 'EEG-F4',
 'EEG-C3',
 'EEG-C4',
 'EEG-P3',
 'EEG-P4',
 'EEG-O1',
 'EEG-O2',
 'EEG-F7',
 'EEG-F8',
 'EEG-P7',
 'EEG-P8',
 'EEG-FZ',
 'EEG-CZ',
 'EEG-PZ',
 'EEG-FC1',
 'EEG-FC2',
 'EEG-CP1',
 'EEG-CP2',
 'EEG-FC5',
 'EEG-FC6',
 'EEG-CP5',
 'EEG-CP6']